In [1]:
import numpy as np
import matplotlib.pyplot as plt
import h5py as h5
import sys
from tqdm import tqdm 
sys.path.append('/home/cosweeney/code/Fits/')

from colossus.cosmology import cosmology
cosmo = cosmology.setCosmology('planck13')
a = 0.83760
redshift=1/a-1
H = cosmo.Hz(redshift)
h = cosmo.h
print(cosmo.h, cosmo.Hz(redshift), cosmo.H0*cosmo.Ez(redshift))

sys.path.append('/home/cosweeney/code/Fits/')

sys.path.append('/home/cosweeney/code/Fits/clive_bias')

from clive.models import *

import matplotlib
matplotlib.rcParams.update({'text.usetex': True, 
                            'font.family': 'Computer Modern Roman'})

from scipy.stats import gennorm, norm, t
import warnings
warnings.filterwarnings('ignore')

from scipy.optimize import curve_fit

import pandas as pd
import os

plt.rcParams['axes.labelsize'] = 20


0.6777 74.71913613009617 74.71913613009617


In [2]:
# compute and inspect the redshift space gg tpcf
from halotools.mock_observables import tpcf, s_mu_tpcf
from halotools.mock_observables import apply_zspace_distortion


def compute_real_tpcf(r, pos, boxsize, num_threads=1, pos_cross=None):
    """
        Computes the real space two point correlation function using halotools
        Args:
                r: np.array
                         binning in pair distances.
                pos: np.ndarray
                         3-D array with the position of the tracers.
                boxsize: float
                        size of the simulation's box.
                num_threads: int
                        number of threads to use.
                pos_cross: np.ndarray
                         3-D array with the position of the tracers
                         (for cross_correlations).

        Returns:
                real_tpcf: np.array
                        1-D array with the real space tpcf.
        """
    if pos_cross is not None:
        do_auto = False
    else:
        do_auto = True
    real_tpcf = tpcf(
        pos,
        r,
        period=boxsize,
        num_threads=num_threads,
        sample2=pos_cross,
        do_auto=do_auto,
    )
    return real_tpcf


def move_to_redshift_space(pos, vel, cosmology, redshift, los_direction, boxsize):
    s_pos = pos.copy()
    z_pos = apply_zspace_distortion(
        true_pos=pos[:, los_direction],
        peculiar_velocity=vel[:, los_direction],
        redshift=redshift,
        cosmology=cosmology,
        Lbox=boxsize,
    )
    # Move tracers to redshift space
    s_pos[:, los_direction] = z_pos
    # Halotools tpcf_s_mu assumes the line of sight is always the z direction
    if los_direction != 2:
        s_pos_old = s_pos.copy()
        s_pos[:, 2] = s_pos_old[:, los_direction]
        s_pos[:, los_direction] = s_pos_old[:, 2]
    return s_pos

def compute_tpcf_s_mu(
    s,
    mu,
    pos,
    vel,
    los_direction,
    cosmology,
    boxsize,
    redshift,
    num_threads=1,
    pos_cross=None,
    vel_cross=None,
):
    """
        Computes the redshift space two point correlation function
        Args:
                s: np.array
                        binning in redshift space pair distances.
                mu: np.array
                         binning in the cosine of the angle respect to the line of sight.
                pos: np.ndarray
                        3-D array with the position of the tracers, in Mpc/h.
                vel: np.ndarray
                         3-D array with the velocities of the tracers, in km/s.
                los_direction: int
                        line of sight direction either 0(=x), 1(=y), 2(=z)
                cosmology: dict
                        dictionary containing the simulatoin's cosmological parameters.
                boxsize:  float
                        size of the simulation's box.
                num_threads: int 
                        number of threads to use.
        Returns:
                tpcf_s_mu: np.ndarray
                        2-D array with the redshift space tpcf.
        """
    if pos_cross is not None:
        do_auto = False
    else:
        do_auto = True

    s_pos = move_to_redshift_space(
        pos, vel, cosmology, redshift, los_direction, boxsize
    )
    if pos_cross is not None and vel_cross is not None:
        s_pos_cross = move_to_redshift_space(
            pos_cross, vel_cross, cosmology, redshift, los_direction, boxsize
        )
    else:
        s_pos_cross = None
    tpcf_s_mu = s_mu_tpcf(
        s_pos,
        s,
        mu,
        period=boxsize,
        estimator=u"Landy-Szalay",
        num_threads=num_threads,
        sample2=s_pos_cross,
        do_auto=do_auto,
    )
    return tpcf_s_mu

In [3]:
from astropy.cosmology import Planck13

print(Planck13)
print(cosmo)

FlatLambdaCDM(name="Planck13", H0=67.77 km / (Mpc s), Om0=0.30712, Tcmb0=2.7255 K, Neff=3.046, m_nu=[0.   0.   0.06] eV, Ob0=0.048252)
Cosmology "planck13" 
    flat = True, Om0 = 0.3071, Ode0 = 0.6928, Ob0 = 0.0483, H0 = 67.77, sigma8 = 0.8288, ns = 0.9611
    de_model = lambda, relspecies = True, Tcmb0 = 2.7255, Neff = 3.0460, powerlaw = False


In [4]:
# %%time
import time

gsb_path = '/spiff/cosweeney/simulations/MDPL2/galaxies/SBp/'
gal_path = '/spiff/cosweeney/simulations/MDPL2/galaxies/UM_full_gal_table.hdf5'

box_length = 1000 #Mpc/h, MDPL2 box side length

#s, mu bins
mu_bins = np.linspace(0,1,120)
rbins = np.linspace(0.001, 80, 60)


with h5.File(gal_path, 'r') as gcat:

    sm_cut = gcat['sm'][()] > 1e10

    gpos = np.array([
        gcat['pos'][:, 0][()][sm_cut],
        gcat['pos'][:, 1][()][sm_cut],
        gcat['pos'][:, 2][()][sm_cut],
    ]).T

    gvel = np.array([
        gcat['pos'][:, 3][()][sm_cut],
        gcat['pos'][:, 4][()][sm_cut], 
        gcat['pos'][:, 5][()][sm_cut],
    ]).T

    frac = 0.01
    idx = np.random.choice(len(gpos), int(frac * len(gpos)), replace=False)

    t0 = time.perf_counter()
    s_mu_tpcf = compute_tpcf_s_mu(
        s=rbins, 
        mu=mu_bins, 
        pos=gpos[idx], 
        vel=gvel[idx], 
        los_direction=2, 
        redshift=redshift,
        cosmology=Planck13, 
        boxsize=box_length,
        num_threads=8
    )
    t1 = time.perf_counter()

    # Rough upper bound: scales as N^2
    n_full, n_sub = len(gpos), len(idx)
    print(f"Subsample took {t1-t0:.1f}s → full estimate: ~{(t1-t0) * (n_full/n_sub)**2:.0f}s")

Subsample took 1.7s → full estimate: ~17096s


In [6]:
24142 / 60 /60

6.706111111111111

In [6]:
17096 / 3600

4.748888888888889

In [5]:
import os
print(os.cpu_count())

32


In [ ]:
from halotools.mock_observables import tpcf_multipole